# Unsupervised Metrics & Visualizations Notebook

This notebook dynamically queries logged unsupervised metrics directly from our local MLflow tracking store (`mlruns/`) and generates publication-quality visual plots comparing our 3 Models (**Z-Score Baseline**, **Isolation Forest**, **LSTM Autoencoder (Ours)**) across our 3 Study Regions (**Altamira MODIS NDVI**, **Brumadinho Sentinel-2 NDWI**, **Mariana Landsat-8 GVMI**).

---

## 📊 Explanation of Evaluated Unsupervised Metrics

Because true pixel-level ground truth labels are rarely available across vast remote sensing time series, we evaluate anomaly detection quality using robust **unsupervised quantitative metrics**:

1. **Spatial Coherence Index (SCP)**
   - **Definition**: Measures the fraction of adjacent horizontal and vertical pixel agreement in anomaly classification maps.
   - **Interpretation**: Higher values (closer to $1.0$) signify physically cohesive spatial event boundaries rather than scattered pixel-level salt-and-pepper noise.

2. **Temporal Entropy ($H$) — Uncertainty Reduction**
   - **Definition**: Computes Shannon entropy ($H = -p \log_2 p - (1-p) \log_2(1-p)$) across predicted state transitions.
   - **Interpretation**: Lower entropy indicates cleaner, less ambiguous predictions where sequence modeling successfully suppresses temporal atmospheric and sensor noise.

3. **Disaster Contrast Ratio (CNR) — Event Sensitivity**
   - **Definition**: The ratio of anomaly detection density during confirmed historical disaster windows compared to baseline non-disaster periods.
   - **Interpretation**: Higher values indicate elevated sensitivity to catastrophic environmental changes (dam failures, mass deforestation) while ignoring routine seasonal variations.

4. **Geographic Cluster Resolution (Average Cluster Size in Pixels)**
   - **Definition**: The mean area (number of contiguous pixels) of connected anomaly clusters computed via 2D connected-component labeling.
   - **Interpretation**: Delineates how models resolve contiguous physical footprint shapes versus diffuse, fragmented spatial noise.

5. **Flicker Ratio (SFR) — Temporal Persistence**
   - **Definition**: The rate of rapid frame-to-frame state oscillations between consecutive timestamps.
   - **Interpretation**: Lower flicker ratios demonstrate smooth temporal continuity and resistance to cloud-induced artifacts.

6. **Execution Time (Seconds) — Computational Efficiency**
   - **Definition**: End-to-end processing and inference runtime measured in seconds per study region.
   - **Interpretation**: Evaluates the scalability trade-off between model sophistication and operational speed.

---


In [ ]:
# Cell 1: Environment & MLflow Dynamic Data Loader
import os
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11

# Initialize MLflow with local mlruns path
local_mlruns = os.path.join(os.getcwd(), 'mlruns').replace('\\', '/')
mlflow.set_tracking_uri(f"file:///{local_mlruns}")
print(f"Tracking URI set to: {mlflow.get_tracking_uri()}")

# Search all experiments and runs dynamically
experiments = mlflow.search_experiments()
exp_ids = [exp.experiment_id for exp in experiments]
df_raw = mlflow.search_runs(experiment_ids=exp_ids)
print(f"Loaded {len(df_raw)} raw runs from MLflow.")

# Clean and extract relevant columns dynamically
records = []
for idx, row in df_raw.iterrows():
    dataset = row.get('params.dataset')
    if pd.isna(dataset) or not dataset:
        run_name = str(row.get('tags.mlflow.runName', ''))
        if 'Altamira' in run_name:
            dataset = 'Altamira'
        elif 'Brumadinho' in run_name:
            dataset = 'Brumadinho'
        elif 'Mariana' in run_name:
            dataset = 'Mariana'
        else:
            continue
            
    model = row.get('params.model_type') or row.get('params.model')
    if pd.isna(model) or not model:
        run_name = str(row.get('tags.mlflow.runName', ''))
        if 'LSTM' in run_name:
            model = 'LSTM Autoencoder (Ours)'
        elif 'Isolation' in run_name:
            model = 'Isolation Forest'
        elif 'ZScore' in run_name or 'Baseline' in run_name:
            model = 'Z-Score Baseline'
        else:
            continue
    else:
        if 'Baseline' in str(model) or 'ZScore' in str(model):
            model = 'Z-Score Baseline'
        elif 'Isolation' in str(model):
            model = 'Isolation Forest'
        elif 'LSTM' in str(model):
            model = 'LSTM Autoencoder (Ours)'

    if dataset == 'Altamira':
        dataset_label = 'Altamira (MODIS NDVI)'
    elif dataset == 'Brumadinho':
        dataset_label = 'Brumadinho (Sentinel-2 NDWI)'
    elif dataset == 'Mariana':
        dataset_label = 'Mariana (Landsat-8 GVMI)'
    else:
        dataset_label = str(dataset)

    scp = row.get('metrics.spatial_coherence_scp')
    entropy_h = row.get('metrics.temporal_entropy_h')
    cnr = row.get('metrics.disaster_contrast_cnr')
    cluster_size = row.get('metrics.avg_cluster_size')
    sfr = row.get('metrics.flicker_ratio_sfr')
    exec_time = row.get('metrics.execution_time_seconds')

    if pd.notna(scp) and pd.notna(entropy_h):
        records.append({
            'dataset_raw': dataset,
            'dataset': dataset_label,
            'model': model,
            'spatial_coherence_scp': float(scp),
            'temporal_entropy_h': float(entropy_h),
            'disaster_contrast_cnr': float(cnr) if pd.notna(cnr) else np.nan,
            'avg_cluster_size': float(cluster_size) if pd.notna(cluster_size) else np.nan,
            'flicker_ratio_sfr': float(sfr) if pd.notna(sfr) else np.nan,
            'execution_time_seconds': float(exec_time) if pd.notna(exec_time) else np.nan
        })

df_metrics = pd.DataFrame(records)
df_summary = df_metrics.groupby(['dataset', 'model'], as_index=False).mean(numeric_only=True)

dataset_order = ['Altamira (MODIS NDVI)', 'Brumadinho (Sentinel-2 NDWI)', 'Mariana (Landsat-8 GVMI)']
model_order = ['Z-Score Baseline', 'Isolation Forest', 'LSTM Autoencoder (Ours)']

df_summary['dataset'] = pd.Categorical(df_summary['dataset'], categories=dataset_order, ordered=True)
df_summary['model'] = pd.Categorical(df_summary['model'], categories=model_order, ordered=True)
df_summary = df_summary.sort_values(['dataset', 'model']).reset_index(drop=True)

print("\n--- Dynamically Extracted Unsupervised Metrics Summary ---")
print(df_summary[['dataset', 'model', 'spatial_coherence_scp', 'temporal_entropy_h', 'disaster_contrast_cnr', 'avg_cluster_size', 'execution_time_seconds']])


In [ ]:
# Cell 2: Plot 1 — Spatial Coherence Index (SCP) Comparison
palette = {'Z-Score Baseline': '#95a5a6', 'Isolation Forest': '#3498db', 'LSTM Autoencoder (Ours)': '#2ecc71'}

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=df_summary,
    x='dataset',
    y='spatial_coherence_scp',
    hue='model',
    palette=palette,
    ax=ax,
    edgecolor='black',
    linewidth=0.8
)

ax.set_title('Spatial Coherence Index (SCP) Across Study Regions', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Spatial Coherence Index (SCP)', fontweight='bold', labelpad=10)
ax.set_ylim(0, 1.25)
# Place legend outside the plot box to avoid overlapping bars
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)

# Add numerical data labels on top of each bar
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.4f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=9.5, fontweight='bold',
                    xytext=(0, 4), textcoords='offset points')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 3: Plot 2 — Noise & Uncertainty Reduction (Temporal Entropy H)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_summary,
    x='dataset',
    y='temporal_entropy_h',
    hue='model',
    palette=palette,
    ax=ax,
    edgecolor='black',
    linewidth=0.8
)

ax.set_title('Uncertainty & Noise Reduction (Temporal Entropy H)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Temporal Entropy H (bits)', fontweight='bold', labelpad=10)
ax.set_ylim(0, max(df_summary['temporal_entropy_h'].dropna()) * 1.25)
ax.legend(title='Model Architecture', frameon=True, facecolor='white', framealpha=0.9)

for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.4f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=9.5, fontweight='bold',
                    xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 4: Plot 3 — Event Sensitivity (Disaster Contrast Ratio - CNR)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_summary,
    x='dataset',
    y='disaster_contrast_cnr',
    hue='model',
    palette=palette,
    ax=ax,
    edgecolor='black',
    linewidth=0.8
)

ax.set_title('Disaster Event Sensitivity (Contrast Ratio - CNR)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Disaster Contrast Ratio (CNR)', fontweight='bold', labelpad=10)
max_cnr = df_summary['disaster_contrast_cnr'].dropna().max()
ax.set_ylim(0, max_cnr * 1.25 if pd.notna(max_cnr) else 5.0)
ax.legend(title='Model Architecture', frameon=True, facecolor='white', framealpha=0.9)

for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.2f}x',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=9.5, fontweight='bold',
                    xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: Plot 4 — Geographic Cluster Resolution (Average Cluster Size - Log Scale)
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=df_summary,
    x='dataset',
    y='avg_cluster_size',
    hue='model',
    palette=palette,
    ax=ax,
    edgecolor='black',
    linewidth=0.8
)

ax.set_yscale('log')
max_val = df_summary['avg_cluster_size'].dropna().max()
ax.set_ylim(1, max_val * 15 if pd.notna(max_val) else 10**7)
ax.set_title('Geographic Cluster Resolution (Avg Anomaly Cluster Size)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Average Spatial Cluster Size (Pixels, Log Scale)', fontweight='bold', labelpad=10)
# Place legend outside the plot box to avoid overlapping bars
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)

for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.1f} px',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=9, fontweight='bold',
                    xytext=(0, 4), textcoords='offset points')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 6: Plot 5 — Efficiency vs Quality Trade-off
fig, ax = plt.subplots(figsize=(10, 6))
markers = {'Z-Score Baseline': 'o', 'Isolation Forest': 's', 'LSTM Autoencoder (Ours)': '^'}

sns.scatterplot(
    data=df_summary,
    x='execution_time_seconds',
    y='spatial_coherence_scp',
    hue='model',
    style='model',
    markers=markers,
    palette=palette,
    s=180,
    ax=ax,
    edgecolor='black',
    alpha=0.9
)

for _, row in df_summary.iterrows():
    if pd.notna(row['execution_time_seconds']) and pd.notna(row['spatial_coherence_scp']):
        ds_short = str(row['dataset']).split()[0]
        ax.annotate(f"{ds_short} ({row['model']})",
                    (row['execution_time_seconds'], row['spatial_coherence_scp']),
                    xytext=(8, -4), textcoords='offset points',
                    fontsize=9, fontweight='bold')

ax.set_title('Efficiency vs Quality Trade-off: Runtime vs Spatial Coherence', fontweight='bold', pad=15)
ax.set_xlabel('Execution Time (Seconds)', fontweight='bold', labelpad=10)
ax.set_ylabel('Spatial Coherence Index (SCP)', fontweight='bold', labelpad=10)
ax.legend(title='Model Architecture', frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()


In [ ]:
# Cell 7: Export Figures
os.makedirs('plots', exist_ok=True)

def save_publication_plot(fig, filename):
    filepath = os.path.join('plots', filename)
    fig.savefig(filepath, dpi=300, bbox_inches='tight')
    print(f"Saved: {filepath}")

# 1. SCP Comparison
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=df_summary, x='dataset', y='spatial_coherence_scp', hue='model', palette=palette, ax=ax, edgecolor='black', linewidth=0.8)
ax.set_title('Spatial Coherence Index (SCP) Comparison', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Spatial Coherence Index (SCP)', fontweight='bold', labelpad=10)
ax.set_ylim(0, 1.25)
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.4f}', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 4), textcoords='offset points')
save_publication_plot(fig, 'plot1_spatial_coherence_scp.png')
plt.close(fig)

# 2. Entropy H
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=df_summary, x='dataset', y='temporal_entropy_h', hue='model', palette=palette, ax=ax, edgecolor='black', linewidth=0.8)
ax.set_title('Uncertainty & Noise Reduction (Temporal Entropy H)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Temporal Entropy H (bits)', fontweight='bold', labelpad=10)
ax.set_ylim(0, max(df_summary['temporal_entropy_h'].dropna()) * 1.3)
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.4f}', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 4), textcoords='offset points')
save_publication_plot(fig, 'plot2_temporal_entropy_h.png')
plt.close(fig)

# 3. Disaster Contrast Ratio (CNR)
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=df_summary, x='dataset', y='disaster_contrast_cnr', hue='model', palette=palette, ax=ax, edgecolor='black', linewidth=0.8)
ax.set_title('Disaster Event Sensitivity (Contrast Ratio - CNR)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Disaster Contrast Ratio (CNR)', fontweight='bold', labelpad=10)
max_cnr = df_summary['disaster_contrast_cnr'].dropna().max()
ax.set_ylim(0, max_cnr * 1.3 if pd.notna(max_cnr) else 5.0)
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.2f}x', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 4), textcoords='offset points')
save_publication_plot(fig, 'plot3_disaster_contrast_cnr.png')
plt.close(fig)

# 4. Cluster Size (Log Scale)
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=df_summary, x='dataset', y='avg_cluster_size', hue='model', palette=palette, ax=ax, edgecolor='black', linewidth=0.8)
ax.set_yscale('log')
max_val = df_summary['avg_cluster_size'].dropna().max()
ax.set_ylim(1, max_val * 15 if pd.notna(max_val) else 10**7)
ax.set_title('Geographic Cluster Resolution (Avg Anomaly Cluster Size)', fontweight='bold', pad=15)
ax.set_xlabel('Study Region & Sensor', fontweight='bold', labelpad=10)
ax.set_ylabel('Average Spatial Cluster Size (Pixels, Log Scale)', fontweight='bold', labelpad=10)
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.1f} px', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=9, fontweight='bold', xytext=(0, 4), textcoords='offset points')
save_publication_plot(fig, 'plot4_cluster_size_log.png')
plt.close(fig)

# 5. Efficiency vs Quality Trade-off
fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(data=df_summary, x='execution_time_seconds', y='spatial_coherence_scp', hue='model', style='model', markers={'Z-Score Baseline': 'o', 'Isolation Forest': 's', 'LSTM Autoencoder (Ours)': '^'}, palette=palette, s=180, ax=ax, edgecolor='black', alpha=0.9)
for _, row in df_summary.iterrows():
    if pd.notna(row['execution_time_seconds']) and pd.notna(row['spatial_coherence_scp']):
        ds_short = str(row['dataset']).split()[0]
        ax.annotate(f"{ds_short} ({row['model']})", (row['execution_time_seconds'], row['spatial_coherence_scp']), xytext=(8, -4), textcoords='offset points', fontsize=9, fontweight='bold')
ax.set_title('Efficiency vs Quality Trade-off: Runtime vs Spatial Coherence', fontweight='bold', pad=15)
ax.set_xlabel('Execution Time (Seconds)', fontweight='bold', labelpad=10)
ax.set_ylabel('Spatial Coherence Index (SCP)', fontweight='bold', labelpad=10)
ax.legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
save_publication_plot(fig, 'plot5_efficiency_vs_quality.png')
plt.close(fig)

print("\nAll 5 publication plots exported successfully to plots/ directory at 300 DPI!")
